In [ ]:
# CS 499 Capstone - Database Enhancement
# Enhanced Grazioso Salvare Search-and-Rescue Dashboard
# Original artifact: CS 340 Project Two
# Enhancement: Secure database configuration, improved query capabilities,
# input validation, and multi-criteria MongoDB searching.

# Dash
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output

# Visualization
import plotly.express as px
import dash_leaflet as dl

# Utilities
import pandas as pd
import re

# Enhanced CRUD module
from animal_shelter import AnimalShelter


#########################
# Database Connection
#########################

# AnimalShelter loads MongoDB configuration from environment variables.
# Database credentials are no longer stored in the notebook source code.
shelter = AnimalShelter()

# Retrieve all animal documents for the initial dashboard view.
df = pd.DataFrame.from_records(shelter.read({}))

# Remove MongoDB ObjectId values before displaying records in Dash.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout
#########################

app = Dash(__name__)

app.layout = html.Div([

    html.Center(
        html.B(
            html.H1("Grazioso Salvare Search-and-Rescue Dashboard")
        )
    ),

    html.P(
        "Enhanced for CS 499",
        style={'fontSize': '18px', 'fontWeight': 'bold'}
    ),

    html.Hr(),

    # Original predefined rescue-type filters.
    html.Div([
        html.H4("Filter by Rescue Type"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {
                    'label': 'Mountain or Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster or Individual Tracking',
                    'value': 'disaster'
                },
                {'label': 'Reset (Show All)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={
                'display': 'inline-block',
                'margin-right': '20px'
            }
        )
    ]),

    html.Hr(),

    # Enhanced multi-criteria MongoDB search.
    html.Div([

        html.H4("Custom Multi-Criteria Search"),

        html.P(
            "Enter any combination of criteria below to search "
            "the animal database."
        ),

        dcc.Input(
            id='search-breed',
            type='text',
            placeholder='Breed',
            debounce=True
        ),

        dcc.Dropdown(
            id='search-sex',
            options=[
                {'label': 'Any Sex', 'value': ''},
                {'label': 'Intact Male', 'value': 'Intact Male'},
                {'label': 'Intact Female', 'value': 'Intact Female'},
                {'label': 'Neutered Male', 'value': 'Neutered Male'},
                {'label': 'Spayed Female', 'value': 'Spayed Female'}
            ],
            value='',
            placeholder='Sex',
            clearable=True
        ),

        dcc.Input(
            id='search-min-age',
            type='number',
            placeholder='Minimum Age (Weeks)',
            min=0
        ),

        dcc.Input(
            id='search-max-age',
            type='number',
            placeholder='Maximum Age (Weeks)',
            min=0
        ),

        html.Div(
            id='search-message',
            style={'marginTop': '10px'}
        )
    ]),

    html.Hr(),

    # Interactive data table.
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],
        data=df.to_dict('records'),
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,
        row_selectable="single",
        selected_rows=[],
        selected_columns=[],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'maxWidth': '250px',
            'whiteSpace': 'normal',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        }
    ),

    html.Br(),
    html.Hr(),

    # Breed visualization and animal-location map.
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6'
            ),
            html.Div(
                id='map-id',
                className='col s12 m6'
            )
        ]
    )
], style={
    'backgroundColor': 'white',
    'color': 'black',
    'padding': '20px'
})


#########################
# Database Query Logic
#########################

def build_custom_query(breed, sex, min_age, max_age):
    """
    Build a MongoDB query from user-supplied search criteria.

    Only criteria supplied by the user are added to the query.
    """

    query = {}

    if breed:
        cleaned_breed = breed.strip()

        if cleaned_breed:
            query['breed'] = {
                '$regex': re.escape(cleaned_breed),
                '$options': 'i'
            }

    if sex:
        query['sex_upon_outcome'] = sex

    age_query = {}

    if min_age is not None:
        age_query['$gte'] = min_age

    if max_age is not None:
        age_query['$lte'] = max_age

    if age_query:
        query['age_upon_outcome_in_weeks'] = age_query

    return query


def build_rescue_query(filter_type):
    """
    Build one of the original predefined rescue-type MongoDB queries.
    """

    if filter_type == 'water':
        return {
            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    if filter_type == 'mountain':
        return {
            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    if filter_type == 'disaster':
        return {
            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }

    return {}


#########################
# Dashboard Controller
#########################

@app.callback(
    [
        Output('datatable-id', 'data'),
        Output('search-message', 'children')
    ],
    [
        Input('filter-type', 'value'),
        Input('search-breed', 'value'),
        Input('search-sex', 'value'),
        Input('search-min-age', 'value'),
        Input('search-max-age', 'value')
    ]
)
def update_dashboard(
    filter_type,
    breed,
    sex,
    min_age,
    max_age
):
    """
    Update the dashboard using either the custom multi-criteria
    search or the original rescue-type filters.
    """

    # Validate the age range before querying MongoDB.
    if (
        min_age is not None
        and max_age is not None
        and min_age > max_age
    ):
        return [], (
            "Minimum age cannot be greater than maximum age."
        )

    custom_query = build_custom_query(
        breed,
        sex,
        min_age,
        max_age
    )

    # Custom criteria take priority when the user supplies them.
    if custom_query:
        query = custom_query
        message = (
            "Custom multi-criteria database search applied."
        )
    else:
        query = build_rescue_query(filter_type)

        if filter_type == 'reset':
            message = "Showing all animals."
        else:
            message = "Rescue-type database filter applied."

    filtered_records = shelter.read(query)

    filtered_df = pd.DataFrame.from_records(filtered_records)

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    if filtered_df.empty:
        return [], "No animals matched the selected criteria."

    return (
        filtered_df.to_dict('records'),
        message
    )


#########################
# Breed Visualization
#########################

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(view_data):
    """
    Display the 15 most common breeds represented in the
    currently filtered data table.
    """

    if not view_data:
        return html.P(
            "No data available to display the chart."
        )

    dff = pd.DataFrame.from_dict(view_data)

    if 'breed' not in dff.columns:
        return html.P(
            "Breed information is unavailable."
        )

    breed_counts = (
        dff['breed']
        .value_counts()
        .nlargest(15)
    )

    fig = px.bar(
        x=breed_counts.index,
        y=breed_counts.values,
        labels={
            'x': 'Breed',
            'y': 'Count'
        },
        title='Top 15 Breeds',
        text_auto=True
    )

    fig.update_layout(
        xaxis_tickangle=-45
    )

    return [
        dcc.Graph(
            figure=fig
        )
    ]


#########################
# Data Table Styling
#########################

@app.callback(
    Output(
        'datatable-id',
        'style_data_conditional'
    ),
    [Input(
        'datatable-id',
        'selected_columns'
    )]
)
def update_styles(selected_columns):
    """
    Highlight selected data-table columns.
    """

    if not selected_columns:
        return []

    return [
        {
            'if': {
                'column_id': column
            },
            'background_color': '#D2F3FF'
        }
        for column in selected_columns
    ]


#########################
# Animal Location Map
#########################

@app.callback(
    Output('map-id', 'children'),
    [
        Input(
            'datatable-id',
            'derived_virtual_data'
        ),
        Input(
            'datatable-id',
            'derived_virtual_selected_rows'
        )
    ]
)
def update_map(view_data, selected_rows):
    """
    Display the location of the animal selected in the table.
    """

    if not view_data:
        return html.P(
            "No animal location available."
        )

    if not selected_rows:
        return html.P(
            "Select an animal in the table to display its location."
        )

    dff = pd.DataFrame.from_dict(view_data)

    row = selected_rows[0]

    if row >= len(dff):
        return html.P(
            "Selected animal is no longer available."
        )

    required_columns = [
        'location_lat',
        'location_long',
        'breed',
        'name'
    ]

    if not all(
        column in dff.columns
        for column in required_columns
    ):
        return html.P(
            "Location information is unavailable."
        )

    animal = dff.iloc[row]

    latitude = animal['location_lat']
    longitude = animal['location_long']

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P(
            "Location information is unavailable for this animal."
        )

    breed = animal['breed']
    name = animal['name']

    return [
        dl.Map(
            style={
                'width': '1000px',
                'height': '500px'
            },
            center=[
                latitude,
                longitude
            ],
            zoom=10,
            children=[
                dl.TileLayer(
                    id='base-layer-id'
                ),
                dl.Marker(
                    position=[
                        latitude,
                        longitude
                    ],
                    children=[
                        dl.Tooltip(
                            str(breed)
                        ),
                        dl.Popup([
                            html.H4(
                                "Animal Name"
                            ),
                            html.P(
                                str(name)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


#########################
# Run Dashboard
#########################

app.run(debug=True)